# Examen Final - Modelos para pronósticos no lineales

- Stonks
- Aissa Berenice
- Tiburivan
- Colomé

## Definición del problema

## Data Loading

In [ ]:
import pandas as pd
import time
from nba_api.stats.endpoints import leaguegamefinder, playergamelog

import plotly.graph_objects as go

In [148]:
# IDs oficiales de la NBA
ID_LAKERS = '1610612747'
ID_LEBRON = '2544'
ID_LUKA = '1629029' # ID oficial de Luka en la NBA

# 1. Obtener TODO el calendario de los Lakers en la temporada 2025-26
game_finder = leaguegamefinder.LeagueGameFinder(team_id_nullable=ID_LAKERS, season_nullable='2025-26')
lakers_games = game_finder.get_data_frames()[0]

# Filtrar solo partidos de temporada regular (eliminar pretemporada si aplica)
lakers_games = lakers_games[lakers_games['MIN'] > 0][['GAME_ID', 'GAME_DATE', 'MATCHUP']].drop_duplicates()
lakers_games = lakers_games[(lakers_games["GAME_DATE"] >= "2025-10-21") & (lakers_games["GAME_DATE"] <= "2026-04-12")]

In [149]:
# 2. Obtener los logs reales de LeBron
time.sleep(0.5) # Pausa para evitar bloqueos de la NBA
lebron_log = playergamelog.PlayerGameLog(player_id=ID_LEBRON, season='2025-26').get_data_frames()[0]
lebron_log = lebron_log[['Game_ID', 'MIN']].rename(columns={'MIN': 'MIN_LeBron', "Game_ID": "GAME_ID"})

In [150]:
# 3. Obtener los logs reales de Luka
time.sleep(0.5)
luka_log = playergamelog.PlayerGameLog(player_id=ID_LUKA, season='2025-26').get_data_frames()[0]
luka_log = luka_log[['Game_ID', 'MIN']].rename(columns={'MIN': 'MIN_Luka', "Game_ID": "GAME_ID"})

In [151]:
# 4. Cruzar el calendario del equipo con los minutos de los jugadores (LEFT JOIN)
df_ts = lakers_games.merge(lebron_log, on='GAME_ID', how='left')
df_ts = df_ts.merge(luka_log, on='GAME_ID', how='left')

df_ts['MIN_LeBron'] = df_ts['MIN_LeBron'].fillna(0)
df_ts['MIN_Luka'] = df_ts['MIN_Luka'].fillna(0)

In [152]:
# 5. Limpieza de datos y conversión a minutos numéricos
def clean_minutes(val):
    if pd.isna(val) or val == 0:
        return 0.0
    # Convertir formato "MM:SS" de la NBA a decimal
    if ':' in str(val):
        parts = str(val).split(':')
        return float(parts[0]) + float(parts[1]) / 60
    return float(val)

df_ts['MIN_LeBron'] = df_ts['MIN_LeBron'].apply(clean_minutes)
df_ts['MIN_Luka'] = df_ts['MIN_Luka'].apply(clean_minutes)

# 6. Crear la variable exógena: ¿Jugó Luka? (1 = Sí, 0 = No, porque sus minutos fueron 0)
df_ts['Luka_Playing'] = df_ts['MIN_Luka'].apply(lambda x: 1 if x > 0 else 0)

# Ordenar cronológicamente para la serie de tiempo
df_ts = df_ts.sort_values('GAME_DATE').reset_index(drop=True)

print(df_ts[['GAME_DATE', 'MATCHUP', 'MIN_LeBron', 'MIN_Luka', 'Luka_Playing']])

     GAME_DATE      MATCHUP  MIN_LeBron  MIN_Luka  Luka_Playing
0   2025-10-21  LAL vs. GSW         0.0      41.0             1
1   2025-10-24  LAL vs. MIN         0.0      35.0             1
2   2025-10-26    LAL @ SAC         0.0       0.0             0
3   2025-10-27  LAL vs. POR         0.0       0.0             0
4   2025-10-29    LAL @ MIN         0.0       0.0             0
..         ...          ...         ...       ...           ...
77  2026-04-05    LAL @ DAL        39.0       0.0             0
78  2026-04-07  LAL vs. OKC         0.0       0.0             0
79  2026-04-09    LAL @ GSW        32.0       0.0             0
80  2026-04-10  LAL vs. PHX        32.0       0.0             0
81  2026-04-12  LAL vs. UTA        17.0       0.0             0

[82 rows x 5 columns]


In [153]:
# 1. Asegurar limpieza de datos
df_ts['GAME_DATE'] = pd.to_datetime(df_ts['GAME_DATE'])
df_ts = df_ts.sort_values('GAME_DATE')

# 2. Crear la figura
fig = go.Figure()

# --- Trace 1: Minutos de LeBron ---
fig.add_trace(go.Scatter(
    x=df_ts['GAME_DATE'],
    y=df_ts['MIN_LeBron'],
    mode='lines+markers',
    name='Minutos LeBron',
    line=dict(color='#552583', width=3), # Púrpura Lakers
    marker=dict(size=7, color='#FDB927'), # Oro Lakers
    hovertemplate='<b>Fecha:</b> %{x}<br><b>LeBron:</b> %{y:.1f} min<extra></extra>'
))

# --- Trace 2: Status de Luka (Variable Exógena) ---
# Multiplicamos por 40 para que la barra suba hasta el promedio de minutos 
# y sea fácil de comparar visualmente con la línea de LeBron
fig.add_trace(go.Scatter(
    x=df_ts['GAME_DATE'],
    y=df_ts['Luka_Playing'] * 45, # Escalamos a 45 para que sirva de fondo
    mode='lines',
    name='¿Jugó Luka?',
    fill='tozeroy', # Crea el efecto de sombreado
    fillcolor='rgba(0, 122, 255, 0.15)', # Azul Luka con transparencia
    line=dict(color='rgba(0, 122, 255, 0.3)', width=1, dash='dot'),
    hovertemplate='<b>Luka en cancha:</b> %{y:0.0f}<extra></extra>'
))

# 3. Personalización del diseño (Sin Slider)
fig.update_layout(
    title=dict(
        text='Impacto de Luka Dončić en los Minutos de LeBron James',
        x=0.5,
        font=dict(size=22)
    ),
    xaxis_title='Calendario Temporada 2025-26',
    yaxis_title='Minutos / Presencia',
    template='plotly_white',
    hovermode='x unified',
    showlegend=True,
    # Eliminamos el slider y el rango selector
    xaxis=dict(
        rangeslider=dict(visible=False),
        type='date'
    ),
    yaxis=dict(range=[0, 50]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()

## Pruebas de Estacionareidad, ACF y PACF

## Escalamiento de datos

## Arquitectura del Modelo

## Entrenamiento del Modelo

## Pronóstico

## Conclusiones